In [1]:
import glob
import re
import os

import numpy as np
import xarray as xr

In [2]:
pattern = os.path.join("./data/SWOT_L3_Expert_v3_0", "SWOT_L3_LR_SSH_Expert_*.nc")
regex = re.compile(r"SWOT_L3_LR_SSH_Expert_\d+_(\d+)_\d{8}T\d{6}_\d{8}T\d{6}_v3\.0(?:\.nc)?")

pass_nums = sorted({regex.match(os.path.basename(f)).group(1)
                    for f in glob.glob(pattern)
                    if regex.match(os.path.basename(f))})

print(pass_nums)

['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018', '019', '020', '021', '022', '023', '024', '025', '026', '027', '028', '151', '153', '155', '158', '159', '161', '162', '165', '166', '167', '168', '170', '174', '178', '180', '181', '182', '183', '186', '188', '189', '190', '191', '192', '193', '195', '196', '200', '201', '202', '203', '206', '208', '209', '213', '214', '215', '216', '218', '219', '220', '221', '222', '224', '225', '226', '227', '228', '230', '232', '233', '236', '237', '238', '239', '241', '243', '244', '245', '248', '249', '250', '251', '253', '254', '257', '259', '262', '265', '266', '268', '270', '271', '272', '275', '276', '277', '278', '281', '282', '283', '285', '286', '287', '288', '289', '290', '292', '293', '294', '297', '298', '299', '300', '301', '302', '303', '305', '306', '307', '309', '310', '313', '318', '319', '320', '321', '322', '327', '328', '329', '330', '331', '332', '333'

In [3]:
def preprocess(ds):
    filename = ds.encoding["source"].split("/")[-1]
    
    if "i_num_line" in ds.data_vars:
        ds = ds.drop_vars(["i_num_line"])
    if "i_num_pixel" in ds.data_vars:
        ds = ds.drop_vars(["i_num_pixel"])

    ds = ds.assign_coords(num_lines=np.arange(ds.sizes["num_lines"]), num_pixels=np.arange(ds.sizes["num_pixels"]))

    cycle_num = int(filename.split("_")[5])
    ds = ds.expand_dims({"cycle_num": [cycle_num]})

    ds = ds.reset_coords(["latitude", "longitude"])

    return ds


def remove_land_only_pixels(ds):
    sea_mask = (~np.isnan(ds["ssha_filtered"])).any(dim="cycle_num")

    min_lat, max_lat = ds["latitude"].where(sea_mask).min(), ds["latitude"].where(sea_mask).max()
    min_lon, max_lon = ds["longitude"].where(sea_mask).min(), ds["longitude"].where(sea_mask).max()
    
    ds = ds.where(
        (
            (ds["latitude"] >= min_lat) & (ds["latitude"] <= max_lat) &
            (ds["longitude"] >= min_lon) & (ds["longitude"] <= max_lon)
        ).compute(),
        drop=True
    )

    return ds


def cleanup_vars(ds):
    ds["time"] = ds.time.max(dim=["num_pixels"], skipna=True).compute()
    ds["latitude"] = ds.latitude.max(dim=["cycle_num"], skipna=True).compute()
    ds["longitude"] = ds.longitude.max(dim=["cycle_num"], skipna=True).compute()
    ds["cross_track_distance"] = ds.cross_track_distance.max(dim=["cycle_num", "num_lines"], skipna=True).compute()
    ds["mdt"] = ds.mdt.max(dim=["cycle_num"], skipna=True).compute()
    ds["mss"] = ds.mss.max(dim=["cycle_num"], skipna=True).compute()

    ds["ssh_filtered"] = ds["ssha_filtered"] + ds["mdt"]

    ds["time"] = ds.time.interpolate_na(
        dim="cycle_num", method="linear", fill_value="extrapolate"
    ).interpolate_na(
        dim="num_lines", method="linear", fill_value="extrapolate"
    )

    ds["latitude"] = ds.latitude.interpolate_na(
        dim="num_lines", method="linear", fill_value="extrapolate"
    ).interpolate_na(
        dim="num_pixels", method="linear", fill_value="extrapolate"
    )

    ds["longitude"] = ds.longitude.interpolate_na(
        dim="num_lines", method="linear", fill_value="extrapolate"
    ).interpolate_na(
        dim="num_pixels", method="linear", fill_value="extrapolate"
    )

    ds = ds.set_coords(["time", "latitude", "longitude"])
    ds = ds.transpose("cycle_num", "num_lines", "num_pixels")

    for var in ds.data_vars:
        if "chunks" in ds[var].encoding:
            del ds[var].encoding["chunks"]
    ds = ds.chunk({"cycle_num": 10, "num_lines": -1, "num_pixels": -1})

    return ds


def open_dataset(pass_num):
    ds = xr.open_mfdataset(
        f"./data/SWOT_L3_Expert_v3_0/SWOT_L3_LR_SSH_Expert_*_{pass_num}_*_*_v3.0.nc", 
        concat_dim="cycle_num",
        combine="nested",
        preprocess=preprocess
    )

    ds = remove_land_only_pixels(ds)
    ds = cleanup_vars(ds)

    return ds

In [ ]:
for pass_num in pass_nums:
    zarr_path = f"./data/SWOT_L3_Expert_v3_0_pass{pass_num}.zarr"
    if not os.path.exists(zarr_path):
        print(f"Processing pass {pass_num}...")
        ds = open_dataset(pass_num)
        ds.to_zarr(zarr_path, mode="w")

Processing pass 003...


/var/folders/xc/bksmt58x2nq8jshz2jbf9b_m0000gn/T/ipykernel_35077/3989092221.py:76: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(
/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Processing pass 016...


/var/folders/xc/bksmt58x2nq8jshz2jbf9b_m0000gn/T/ipykernel_35077/3989092221.py:76: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(
/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Processing pass 151...


...

/var/folders/xc/bksmt58x2nq8jshz2jbf9b_m0000gn/T/ipykernel_35077/3989092221.py:76: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(
/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/var/folders/xc/bksmt58x2nq8jshz2jbf9b_m0000gn/T/ipykernel_35077/3989092221.py:76: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data

Processing pass 203...


/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/var/folders/xc/bksmt58x2nq8jshz2jbf9b_m0000gn/T/ipykernel_35077/3989092221.py:76: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


Processing pass 206...


/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
